# CUDA DPO Alignment — Veritas Verifier (Kaggle/Colab)

This notebook aligns the CUDA QLoRA adapter (`checkpoints/cuda_qlora_verifier/`, from `notebooks/12_cuda_qlora_kaggle_colab.ipynb`) with **Direct Preference Optimization (DPO)** using TRL's `DPOTrainer` and `data/processed/preference_pairs.jsonl` (1,382 `{"prompt", "chosen", "rejected"}` pairs built by `scripts/build_preference_pairs_real.py`).

**Prerequisite**: `checkpoints/cuda_qlora_verifier/` must already exist (run `notebooks/12_cuda_qlora_kaggle_colab.ipynb` first and copy its outputs back into the repo before running this notebook).

It requires a **CUDA GPU** (Kaggle: Settings > Accelerator > GPU T4 x2, or Colab: Runtime > Change runtime type > GPU).

Run the cells in order:
1. Check GPU availability.
2. Install dependencies.
3. Clone the Veritas repo and upload the QLoRA adapter (`checkpoints/cuda_qlora_verifier/`).
4. Run DPO training, which evaluates the adapter **before** and **after** DPO on `data/processed/mlx_lora/valid.jsonl`.
5. Zip the resulting artifacts and download them.

**Outputs to copy back into the repo** (see the final cell):
- `checkpoints/cuda_dpo_verifier/` (DPO-aligned LoRA adapter files)
- `reports/cuda_dpo_eval.json` / `reports/cuda_dpo_eval.md` (before/after metrics: verdict accuracy, macro F1, citation valid rate, unsupported-sentence rate, verdict consistency rate)

Per project rules, DPO is only considered complete once these files exist from an actual run of this notebook — do not fabricate them.

## 1. Check GPU availability

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "This notebook requires a CUDA GPU. On Kaggle: Settings > Accelerator > "
        "GPU T4 x2. On Colab: Runtime > Change runtime type > GPU."
    )

## 2. Install dependencies

In [ ]:
!pip install -q -U torch transformers peft bitsandbytes accelerate datasets trl

## 3. Clone the Veritas repo and add the QLoRA adapter

`data/processed/preference_pairs.jsonl` and `data/processed/mlx_lora/` are already tracked in the repo. `checkpoints/cuda_qlora_verifier/` is gitignored, so upload it from your Phase 4 run (`cuda_qlora_artifacts.zip`) and unzip it into `Veritas/checkpoints/cuda_qlora_verifier/`.

In [ ]:
import os

if not os.path.exists("Veritas"):
    !git clone https://github.com/sushildalavi/veritas.git Veritas
%cd Veritas

In [ ]:
# Upload cuda_qlora_artifacts.zip (from notebooks/12_cuda_qlora_kaggle_colab.ipynb) and unzip it here.
# On Colab:
#   from google.colab import files
#   uploaded = files.upload()  # select cuda_qlora_artifacts.zip
# On Kaggle, add cuda_qlora_artifacts.zip as a notebook input dataset instead.

!unzip -o cuda_qlora_artifacts.zip -d .
!ls checkpoints/cuda_qlora_verifier

## 4. Run DPO training and before/after evaluation

Loads `checkpoints/cuda_qlora_verifier/` as the starting policy, evaluates it on `data/processed/mlx_lora/valid.jsonl` (**before**), runs `trl.DPOTrainer` on `data/processed/preference_pairs.jsonl`, saves the result to `checkpoints/cuda_dpo_verifier/`, then evaluates again (**after**).

In [ ]:
!python3 scripts/train_cuda_dpo.py \
    --base-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --qlora-adapter-path checkpoints/cuda_qlora_verifier \
    --preference-pairs-file data/processed/preference_pairs.jsonl \
    --output-dir checkpoints/cuda_dpo_verifier \
    --report-json reports/cuda_dpo_eval.json \
    --report-md reports/cuda_dpo_eval.md

In [ ]:
import json

with open("reports/cuda_dpo_eval.json") as handle:
    report = json.load(handle)
print("before:", report["before"])
print("after: ", report["after"])
print("delta: ", report["delta"])

## 5. Zip artifacts and download

In [ ]:
!zip -r cuda_dpo_artifacts.zip \
    checkpoints/cuda_dpo_verifier \
    reports/cuda_dpo_eval.json \
    reports/cuda_dpo_eval.md

In [ ]:
try:
    from google.colab import files
    files.download("cuda_dpo_artifacts.zip")
except ImportError:
    print(
        "Not running in Colab. On Kaggle, find cuda_dpo_artifacts.zip in "
        "/kaggle/working/Veritas and download it from the notebook's Output tab."
    )

## Files to copy back into the Veritas repo

After unzipping `cuda_dpo_artifacts.zip`, copy these paths into your local clone of the repo (preserving the directory structure):

- `checkpoints/cuda_dpo_verifier/` → `checkpoints/cuda_dpo_verifier/`
- `reports/cuda_dpo_eval.json` → `reports/cuda_dpo_eval.json`
- `reports/cuda_dpo_eval.md` → `reports/cuda_dpo_eval.md`

Once these exist, CUDA DPO can be marked complete and `reports/cuda_dpo_eval.md`'s before/after table can be cited in the project docs.